# Archivo XML
Un XML está compuesto de tags y atributos. Es posible leer desde un archivo local o a través de una URL mediante librerías como `urllib`.

In [1]:
import xml.etree.ElementTree as ET
tree = ET.parse('ejemplo.xml')
print(type(tree))
root = tree.getroot()
root

<class 'xml.etree.ElementTree.ElementTree'>


<Element 'data' at 0x000001442CBFF2E0>

Dentro de root, hay n elementos, considerados como los *children* de root. Cada elemento del XML tiene un *tag* y varios atributos en formato clave-valor.

In [2]:
for child in root:
    print(child.tag)
    print(child.attrib)
    print(child.attrib.get('name') + '\n')

country
{'name': 'Liechtenstein'}
Liechtenstein

country
{'name': 'Singapore'}
Singapore

country
{'name': 'Panama'}
Panama



Para obtener los children de cada línea:

In [3]:
for node in root.iter():
    print(node.tag)

data
country
rank
year
gdppc
neighbor
neighbor
country
rank
year
gdppc
neighbor
country
rank
year
gdppc
neighbor
neighbor


Los elementos van ordenados, por lo que podremos acceder a los mismos mediante su orden.

Si queremos acceder a lo que hay dentro de los tags: `.text`. Devuelve un String, aunque estemos hablando de un número.

In [4]:
print(root[0][1])
print(root[0][1].text)

<Element 'year' at 0x000001442CBFF3D0>
2008


Para iterar sobre unos elementos concretos. Podría haber atributos que se llamen igual en diferentes puntos del XML, y quizá ese no es el comportamiento que deseamos. Podemos filtrar los que nos interesan usando expresiones XPath.

In [5]:
for neighbor in root.iter('neighbor'):
    print(type(neighbor))
    print(neighbor.attrib)
    print(neighbor.attrib['name'])

<class 'xml.etree.ElementTree.Element'>
{'name': 'Austria', 'direction': 'E'}
Austria
<class 'xml.etree.ElementTree.Element'>
{'name': 'Switzerland', 'direction': 'W'}
Switzerland
<class 'xml.etree.ElementTree.Element'>
{'name': 'Malaysia', 'direction': 'N'}
Malaysia
<class 'xml.etree.ElementTree.Element'>
{'name': 'Costa Rica', 'direction': 'W'}
Costa Rica
<class 'xml.etree.ElementTree.Element'>
{'name': 'Colombia', 'direction': 'E'}
Colombia


1. `find` para buscar un tag dentro del árbol inmediato, y no de sus hijos.
2. `iter` para buscar un tag dentro de todo un árbol, incluidos sus hijos.
3. `get` para conseguir el valor de un atributo.

In [6]:
for country in root.findall('country'):
    rank = country.find('rank').text
    neighbor = country.find('neighbor').text
    name = country.get('name')
    
    print(name, neighbor, rank)

Liechtenstein None 1
Singapore None 4
Panama None 68


También podríamos modificarlo

In [7]:
for rank in root.iter('rank'):
    new_rank = int(rank.text) + 1
    rank.text = str(new_rank)
    rank.set('updated', 'yes')

In [8]:
for country in root.findall('country'):
    rank = country.find('rank').text 
    name = country.get('name') 
    print(name, rank)

Liechtenstein 2
Singapore 5
Panama 69


In [9]:
tree.write('output.xml')

O eliminar elementos

In [10]:
for country in root.findall('country'):
    rank = int(country.find('rank').text)
    if rank > 50:
        root.remove(country)

Para escribir los datos en un nuevo XML

In [11]:
tree.write('output.xml')

Cada elemento que encuentra `findall()` o `iter()` es un `Element`, sobre el que podemos acceder a sus datos gracias a atributos como como `tag` o `attrib`.

Acceso mediante XPath. Ya que la estructura de los XML es un árbol de tags, podremos acceder a los elementos mediante la ruta relativa de los tags.

In [12]:
print(root.findall('./country/neighbor'))

[<Element 'neighbor' at 0x000001442CBFF470>, <Element 'neighbor' at 0x000001442CBFF4C0>, <Element 'neighbor' at 0x000001442CBFF650>]


# XML con RSS
RSS es una manera que tienen las páginas web de publicar su contenido. En este caso no es un HTML, ni un CSS como se hace habitualmente, sino que será un XML, con un árbol de tags y distinta información. Páginas que utilizan esto son periódicos, foros o blogs. Permite acceder a los titulares de noticias, tanto de las generales, como de las secciones del periódico, de tal manera que puedas monitorizarlos en una aplicación. Los datos son abiertos y el formato de publicación es XML.

En nuestro caso vamos a desarrollar un programa mediante el que recogeremos el RSS del periódico *El Pais* y montaremos una tabla con las principales noticias.

In [13]:
from urllib.request import urlopen
from xml.etree.ElementTree import parse

var_url = urlopen('https://feeds.elpais.com/mrss-s/pages/ep/site/elpais.com/section/ultimas-noticias/portada')
# var_url = urlopen('http://ep00.epimg.net/rss/tags/ultimas_noticias.xml')
xmldoc = parse(var_url)

for item in xmldoc.findall('./channel/item'):
    title = item.findtext('title')
    date = item.findtext('pubDate')
    link = item.findtext('link')
    
    
    print("título: ", title, "\nfecha:", date, "\nenlace:", link, "\n")


título:  Irán mantiene la amenaza en el estrecho de Ormuz pese a abrir el paso a algunos buques de países no enemigos 
fecha: Wed, 18 Mar 2026 04:30:01 GMT 
enlace: https://elpais.com/internacional/2026-03-18/iran-mantiene-la-amenaza-en-el-estrecho-de-ormuz-pese-a-abrir-el-paso-a-algunos-buques-de-paises-no-enemigos.html 

título:  El CIS detecta una caída de Vox de más de dos puntos en el barómetro de marzo 
fecha: Wed, 18 Mar 2026 11:37:40 GMT 
enlace: https://elpais.com/espana/2026-03-18/el-cis-detecta-una-caida-de-vox-de-mas-de-dos-puntos-en-el-barometro-de-marzo.html 

título:  ‘Efecto Trump’, voto útil, “pleamar” electoral: en busca del porqué del éxito a medias de Vox este 15-M 
fecha: Wed, 18 Mar 2026 04:30:01 GMT 
enlace: https://elpais.com/espana/2026-03-18/efecto-trump-voto-util-pleamar-electoral-en-busca-del-porque-del-exito-a-medias-de-vox-este-15-m.html 

título:  Elisa Mouliaá: “Puede que haya sido torpe, pero yo no he mentido” 
fecha: Wed, 18 Mar 2026 04:30:01 GMT 
enla

Enlaces documentación

https://docs.python.org/3/library/xml.etree.elementtree.html

https://rico-schmidt.name/pymotw-3/xml.etree.ElementTree/parse.html